# Part 2b - NLP Feature Engineering with spaCy

> Case study part: Part 2 of 3 (second of two notebooks)
> Required for: All students (3-credit and 4-credit)
> Role: You are a Real Estate Analyst at Quad Capital Partners (QCP).

The Part 2a baseline used only structured tabular features. But almost every
listing also has a free-text description written by the listing agent (about
130 words at the median) - the field that brags about the granite
countertops, the new roof, the walkable distance to campus, and the basement
that's "perfect for a graduate-student rental."

In this notebook we use spaCy to turn that free text into structured
features and feed them back into the same modeling pipeline from Part 2a.
The target is still `price`, the most recent sold price. The goal is an
honest test, on the same held-out test set, of whether the text improves the
model.

:::{tip} Built for speed

Running a full spaCy pipeline (tagger + parser + lemmatizer) over 35,000
descriptions can take several minutes. This notebook deliberately keeps
spaCy lightweight: we only use the tokenizer for the amenity matcher,
and we let `TfidfVectorizer` handle text normalization for the
bag-of-words features. The amenity matcher runs in seconds.

:::

## ⚙️ Setup


:::{note} Setup

This notebook needs `spacy` and the `en_core_web_sm` model. Both are listed in
`requirements.txt` at the repository root, so
`python -m pip install -r requirements.txt` installs everything.

:::

In [1]:
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.io as pio

pio.templates.default = "simple_white"

import spacy
from spacy.matcher import PhraseMatcher

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBRegressor

pd.set_option("display.max_columns", 80)

Load spaCy with every pipeline component disabled, so only the tokenizer
remains. This is roughly an order of magnitude faster than running the full
pipeline because we skip the token-to-vector layer, the part-of-speech
tagger, the dependency parser, the named-entity recognizer, and the
lemmatizer.

In [2]:
nlp = spacy.load(
    "en_core_web_sm",
    disable=["tok2vec", "tagger", "parser", "ner", "lemmatizer", "attribute_ruler"],
)
print("spaCy model loaded:", nlp.meta["name"], nlp.meta["version"])
print("Active pipeline components:", nlp.pipe_names)

spaCy model loaded: core_web_sm 3.8.0
Active pipeline components: []


:::{note} Why tokenizer-only?

All we need from spaCy in this notebook is its rule-based English tokenizer
for the `PhraseMatcher`. We don't need part-of-speech tags or dependency
trees, so we disable everything else (the empty list above confirms it).
`TfidfVectorizer` will handle tokenization for the bag-of-words text features
on its own.

:::

## 📥 Load the Cleaned Data and the Part 2a Split


In [3]:
DATA_DIR = "data"

df = pd.read_parquet(f"{DATA_DIR}/properties_clean.parquet")
splits = pd.read_parquet(f"{DATA_DIR}/baseline_split.parquet")

train_idx = splits.loc[splits["split"] == "train", "row_index"].values
test_idx = splits.loc[splits["split"] == "test", "row_index"].values

print(f"{len(df):,} listings, {len(train_idx):,} train, {len(test_idx):,} test")


35,273 listings, 28,218 train, 7,055 test


## 🔎 A Quick Look at What Listing Agents Actually Write


In [4]:
sample = df.dropna(subset=["description"]).sample(n=3, random_state=7)
for _, row in sample.iterrows():
    print(
        f"--- {row['address_street']}, {row['market']}  (last sold ${row['price']:,}) ---"
    )
    print(row["description"][:400], "...\n")

--- 2631 Overland Passage Rd, Chapel Hill, NC  (last sold $480,000) ---
A stunning light filled home on a desirable corner lot in the perennial favorite neighborhood of The Trails. This house was almost entirely rebuilt following a fire in 2016/17 by Kovens Builders.  New windows, Roof, HVAC, ducts, electrical etc.  See docs for details.  New kitchen cabinetry and granite counters (2020), hardwood floors throughout the downstairs.   Freshly painted.       But truly th ...

--- 145 Lullwater Rd, Athens, GA  (last sold $741,500) ---
This wonderful split foyer mid-century home in the Glenwood neighborhood of Five Points is full of charm and has loads of space. This 4 bedroom, 3.5 bathroom home features 3009 square feet of living space. From the entry going up, you will find a large living area with a fireplace. Beyond the living area, you will find a dining room and kitchen that overlook the beautiful backyard and covered porc ...

--- 526 S 26th St, Lafayette, IN  (last sold $180,000) --

## 📏 Lightweight Length Features

Description length is itself a candidate price signal: Part 1 showed that the
median price rises from the shortest fifth of descriptions to the longest.
We compute these from raw text - no spaCy needed.

In [5]:
desc = df["description"].fillna("")
length_feats = pd.DataFrame(
    {
        "n_chars": desc.str.len(),
        "n_words": desc.str.split().map(len),
    }
)
length_feats.describe().round(1)


,n_chars,n_words
count,35273.0,35273.0
mean,887.1,142.1
std,551.3,87.2
min,0.0,0.0
25%,526.0,85.0
50%,791.0,127.0
75%,1139.0,182.0
max,8102.0,1295.0


## 🏷️ Amenity Detection with `PhraseMatcher`

Listing agents reuse a small vocabulary of "selling words." We define an
amenity dictionary, compile it into a spaCy `PhraseMatcher`, and emit a
binary flag per amenity for every listing. These are the kinds of features
a hedonic pricing model loves.


Define the amenity dictionary. The matcher compares tokens exactly (after
lowercasing), so "granite countertop" does not match "granite countertops".
We therefore list the plural form of each noun phrase too. An alternative is
to match on lemmas (`attr="LEMMA"`), but that needs the tagger and lemmatizer,
which would make the notebook much slower.

In [6]:
AMENITIES = {
    "granite": [
        "granite countertop",
        "granite countertops",
        "granite counter",
        "granite counters",
    ],
    "stainless": [
        "stainless steel appliance",
        "stainless steel appliances",
        "stainless appliance",
        "stainless appliances",
    ],
    "hardwood": ["hardwood floor", "hardwood floors", "hardwood flooring"],
    "updated_kitchen": [
        "updated kitchen",
        "renovated kitchen",
        "remodeled kitchen",
        "new kitchen",
    ],
    "open_floor_plan": ["open floor plan", "open concept"],
    "primary_suite": ["primary suite", "master suite", "owner suite"],
    "finished_basement": ["finished basement", "walkout basement"],
    "two_car_garage": ["two car garage", "2 car garage", "two-car garage"],
    "fenced_yard": ["fenced yard", "fenced backyard", "privacy fence"],
    "deck_or_patio": [
        "deck",
        "decks",
        "patio",
        "patios",
        "screened porch",
        "screened porches",
    ],
    "new_roof": ["new roof", "newer roof", "roof replaced"],
    "new_hvac": ["new hvac", "new furnace", "new ac", "new air conditioner"],
    "near_campus": [
        "walking distance to campus",
        "close to campus",
        "near campus",
        "minutes to campus",
    ],
    "investor": [
        "investor",
        "investors",
        "rental property",
        "investment property",
        "rental income",
    ],
    "fixer_upper": ["fixer upper", "needs work", "as is", "tlc"],
    "luxury": ["luxury", "high end", "designer", "custom built"],
}

matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
for label, phrases in AMENITIES.items():
    matcher.add(label, [nlp.make_doc(p) for p in phrases])

Run the matcher with the tokenizer-only pipeline. We use `nlp.tokenizer.pipe`
directly so spaCy runs nothing but the tokenizer. This is the single largest
speedup in the notebook.

In [7]:
def amenity_flags(texts, batch_size=500):
    label_lookup = {nlp.vocab.strings[label]: label for label in AMENITIES}
    cols = [f"amen_{label}" for label in AMENITIES]
    rows = np.zeros((len(texts), len(cols)), dtype=np.int8)
    col_idx = {name: i for i, name in enumerate(cols)}

    docs = nlp.tokenizer.pipe(texts, batch_size=batch_size)
    for i, doc in enumerate(docs):
        for match_id, _, _ in matcher(doc):
            rows[i, col_idx[f"amen_{label_lookup[match_id]}"]] = 1
    return pd.DataFrame(rows, columns=cols)


In [8]:
texts = df["description"].fillna("").tolist()
amen = amenity_flags(texts)
amen.head(3)


,amen_granite,amen_stainless,amen_hardwood,amen_updated_kitchen,amen_open_floor_plan,amen_primary_suite,amen_finished_basement,amen_two_car_garage,amen_fenced_yard,amen_deck_or_patio,amen_new_roof,amen_new_hvac,amen_near_campus,amen_investor,amen_fixer_upper,amen_luxury
0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0


Share of listings mentioning each amenity.


In [9]:
amen_share = amen.mean().sort_values(ascending=False).mul(100).round(1)
amen_share.to_frame("pct_listings_mentioning")


,pct_listings_mentioning
amen_deck_or_patio,44.7
amen_hardwood,19.6
amen_primary_suite,19.3
amen_stainless,16.5
amen_fenced_yard,14.1
amen_granite,12.1
amen_luxury,11.9
amen_new_roof,11.6
amen_open_floor_plan,11.4
amen_updated_kitchen,10.7


In [10]:
share_df = amen_share.sort_values().reset_index()
share_df.columns = ["amenity", "pct"]
fig = px.bar(
    share_df,
    x="pct",
    y="amenity",
    orientation="h",
    title="Share of campus-town listings mentioning each amenity",
    labels={"pct": "% of listings", "amenity": ""},
    height=500,
)
fig.show()


:::{caution} Check your matcher's coverage

An earlier version of this notebook listed only singular phrases ("granite
countertop", "hardwood floor"). It flagged granite in 1.5% of listings,
stainless appliances in 0.3%, and hardwood floors in 3.3%, because agents
almost always write the plural. Before you trust a keyword feature, read a few
descriptions and compare the match rate with what you see.

:::

## 💵 Do Amenity Mentions Move Price?

A quick before-modeling sanity check: the median last sold price for listings
that mention each amenity vs. listings that do not.

In [11]:
combo = pd.concat(
    [df.reset_index(drop=True)[["price"]], amen.reset_index(drop=True)], axis=1
)
lift = []
for c in amen.columns:
    with_a = combo.loc[combo[c] == 1, "price"].median()
    without_a = combo.loc[combo[c] == 0, "price"].median()
    lift.append(
        {
            "amenity": c.replace("amen_", ""),
            "median_with": with_a,
            "median_without": without_a,
            "pct_lift": (with_a - without_a) / without_a * 100,
        }
    )
lift_df = pd.DataFrame(lift).sort_values("pct_lift", ascending=False)
lift_df


,amenity,median_with,median_without,pct_lift
5,primary_suite,525000.0,390000.0,34.615385
15,luxury,535900.0,400000.0,33.975000
6,finished_basement,500000.0,403500.0,23.915737
0,granite,485000.0,400000.0,21.250000
9,deck_or_patio,450000.0,380000.0,18.421053
2,hardwood,465000.0,400000.0,16.250000
1,stainless,427000.0,407000.0,4.914005
4,open_floor_plan,427750.0,409000.0,4.584352
3,updated_kitchen,420000.0,410000.0,2.439024
7,two_car_garage,405000.0,411000.0,-1.459854


In [12]:
fig = px.bar(
    lift_df.sort_values("pct_lift"),
    x="pct_lift",
    y="amenity",
    orientation="h",
    color="pct_lift",
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    title="Naive price lift when each amenity is mentioned (median, %)",
    labels={
        "pct_lift": "Median price lift vs. listings without the phrase",
        "amenity": "",
    },
    height=500,
)
fig.show()


:::{caution} Confounded, but useful

This chart is correlational. Listings that mention "luxury" have a median
price about 34% higher than listings that do not, partly because genuinely
expensive houses are described that way - not because adding the word makes a
house worth more. The same logic runs the other way for `investor` and
`fixer_upper`. The proper test is whether these features improve the
predictive model, which is what we do next.

:::

## 🧱 Build the NLP-Augmented Modeling Dataset


Combine cleaned tabular features with the new NLP-derived features.


In [13]:
features_full = pd.concat(
    [
        df.reset_index(drop=True),
        length_feats.reset_index(drop=True),
        amen.reset_index(drop=True),
    ],
    axis=1,
)
features_full["log_price"] = np.log1p(features_full["price"])


Define the feature lists. Note the new NLP-derived numeric and text columns.


In [14]:
NUMERIC_FEATURES = [
    "beds",
    "baths",
    "living_area",
    "lot_size",
    "year_built",
    "reso_parking_capacity",
    "latitude",
    "longitude",
    # New: NLP-derived numeric signals
    "n_chars",
    "n_words",
] + list(amen.columns)

CATEGORICAL_FEATURES = [
    "address_state",
    "address_city",
    "address_zipcode",
    "campus",
    "reso_architectural_style",
]
TEXT_FEATURE = "description"


Assemble the modeling matrix and re-use the same train/test split as Part 2a
so the comparison is apples-to-apples.


In [15]:
X = features_full[NUMERIC_FEATURES + CATEGORICAL_FEATURES + [TEXT_FEATURE]].copy()
X["address_zipcode"] = X["address_zipcode"].astype(str)
X[TEXT_FEATURE] = X[TEXT_FEATURE].fillna("")

y_log = features_full["log_price"]
y_raw = features_full["price"]

X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train_log, y_test_log = y_log.loc[train_idx], y_log.loc[test_idx]
y_train_raw, y_test_raw = y_raw.loc[train_idx], y_raw.loc[test_idx]
print(f"Train: {len(X_train):,}   Test: {len(X_test):,}")


Train: 28,218   Test: 7,055


## 🔧 Pipeline = Tabular + TF-IDF on Raw Description

We let `TfidfVectorizer` do its own tokenization and stopword removal
directly on the raw description text. This is much faster than running
every description through spaCy first.

In [16]:
numeric_pipeline = Pipeline(
    [
        ("impute", SimpleImputer(strategy="median")),
    ]
)
categorical_pipeline = Pipeline(
    [
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=20)),
    ]
)
text_pipeline = TfidfVectorizer(
    max_features=2_000,
    ngram_range=(1, 2),
    min_df=10,
    stop_words="english",
    lowercase=True,
)


In [17]:
preprocessor = ColumnTransformer(
    [
        ("num", numeric_pipeline, NUMERIC_FEATURES),
        ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
        ("txt", text_pipeline, TEXT_FEATURE),
    ]
)

nlp_model = Pipeline(
    [
        ("preprocess", preprocessor),
        (
            "regressor",
            XGBRegressor(
                n_estimators=600,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.9,
                colsample_bytree=0.9,
                random_state=42,
                n_jobs=-1,
                tree_method="hist",
            ),
        ),
    ]
)

nlp_model.fit(X_train, y_train_log)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](32,)","['beds','baths','living_area',...,'campus','reso_architectural_style', 'description']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,32
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. T

## 📏 Compare Against the Baseline on the Same Test Set


Helper for the comparison table.


In [18]:
def metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "MAPE": np.mean(np.abs((y_true - y_pred) / y_true)) * 100,
    }


Predict with the NLP model, then re-train a baseline (no NLP features) on
the same split for a fair comparison.


In [19]:
nlp_pred_test = np.expm1(nlp_model.predict(X_test))


In [20]:
baseline_num = [
    c
    for c in NUMERIC_FEATURES
    if not c.startswith("amen_") and c not in {"n_chars", "n_words"}
]
baseline_pre = ColumnTransformer(
    [
        ("num", numeric_pipeline, baseline_num),
        ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
    ]
)
baseline_model = Pipeline(
    [
        ("preprocess", baseline_pre),
        (
            "regressor",
            XGBRegressor(
                n_estimators=600,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.9,
                colsample_bytree=0.9,
                random_state=42,
                n_jobs=-1,
                tree_method="hist",
            ),
        ),
    ]
)
baseline_model.fit(X_train, y_train_log)
baseline_pred_test = np.expm1(baseline_model.predict(X_test))


In [21]:
comparison = pd.DataFrame(
    {
        "Baseline (Part 2a)": metrics(y_test_raw, baseline_pred_test),
        "Baseline + spaCy NLP": metrics(y_test_raw, nlp_pred_test),
    }
).T
comparison


,MAE,RMSE,R2,MAPE
Baseline (Part 2a),73863.609375,148555.980560,0.802019,17.080081
Baseline + spaCy NLP,72947.273438,149785.249154,0.798729,16.481231


:::{tip} Read the table the way the Investment Committee will

The result is mixed. MAPE fell from about 17.1% to 16.5% and MAE fell by
about \$900, but RMSE rose slightly and R² fell slightly (0.802 to 0.799).
The text features lower the average percentage error a little, but they do
not reduce the large-dollar misses that dominate RMSE. Present it that way:
quote every metric and its change ("MAPE fell from X% to Y%, while RMSE rose
from A to B"), not only the one that improved.

:::

## 🧪 Where Did NLP Help Most?

We expect NLP to help the most where the tabular features alone are weakest,
for example where a home's value depends on condition and amenities, not just
size and location. Here we break the change in error down by market.

In [22]:
diag = pd.DataFrame(
    {
        "actual": y_test_raw.values,
        "baseline": baseline_pred_test,
        "nlp": nlp_pred_test,
        "market": features_full.loc[test_idx, "market"].values,
    }
)
diag["baseline_pct_err"] = (
    (diag["actual"] - diag["baseline"]).abs() / diag["actual"] * 100
)
diag["nlp_pct_err"] = (diag["actual"] - diag["nlp"]).abs() / diag["actual"] * 100
diag["improvement_pp"] = diag["baseline_pct_err"] - diag["nlp_pct_err"]

by_market = (
    diag.groupby("market", as_index=False)
    .agg(
        n=("actual", "count"),
        mean_baseline=("baseline_pct_err", "mean"),
        mean_nlp=("nlp_pct_err", "mean"),
        improvement_pp=("improvement_pp", "mean"),
    )
    .sort_values("improvement_pp", ascending=False)
)
by_market.round(2)


,market,n,mean_baseline,mean_nlp,improvement_pp
7,"Gainesville, FL",1125,22.94,20.64,2.30
14,"Urbana, IL",218,19.06,16.98,2.07
3,"Champaign, IL",471,20.86,19.19,1.67
2,"Carrboro, NC",43,14.99,14.25,0.74
6,"Davis, CA",190,13.01,12.43,0.57
4,"Chapel Hill, NC",547,16.10,15.54,0.57
1,"Athens, GA",486,20.32,19.77,0.55
9,"Lafayette, IN",612,13.05,12.67,0.38
10,"Madison, WI",1025,13.55,13.17,0.37
11,"Middleton, WI",124,12.73,12.54,0.19


In [23]:
fig = px.bar(
    by_market.sort_values("improvement_pp"),
    x="improvement_pp",
    y="market",
    orientation="h",
    color="improvement_pp",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    title="Reduction in mean |% error| from adding NLP features (percentage points)",
    labels={
        "improvement_pp": "Δ MAPE (baseline - NLP), percentage points",
        "market": "",
    },
    height=600,
)
fig.show()


## 💾 Persist the NLP Features for Part 3

4-credit students will reuse the length features and amenity flags in Part 3
alongside the price-history features. We save them to Parquet so Part 3
does not have to re-run the matcher over 35,000 descriptions.


In [24]:
nlp_pack = pd.concat(
    [
        df[["property_id"]].reset_index(drop=True),
        length_feats.reset_index(drop=True),
        amen.reset_index(drop=True),
    ],
    axis=1,
)
nlp_pack.to_parquet(f"{DATA_DIR}/nlp_features.parquet", index=False)
print(
    f"Saved {len(nlp_pack):,} rows of NLP features -> {DATA_DIR}/nlp_features.parquet"
)


Saved 35,273 rows of NLP features -> data/nlp_features.parquet


## 🧠 Investment-Committee Takeaways (Part 2b)

What this notebook shows, based on the outputs above:

- The amenity matcher needs plural phrases. With them, granite, stainless,
  and hardwood mentions appear in about 12%, 17%, and 20% of listings.
- The largest naive price lifts are for `primary_suite`, `luxury`,
  `finished_basement`, and `granite` (a median price about 21% to 35% higher).
  Listings that mention `investor` or `fixer_upper` have a median price about
  40% lower. These are correlations, not causal effects.
- Adding the text features gives a mixed result on the same held-out split:
  MAPE and MAE fell slightly, while RMSE and R² got slightly worse.
- The gain is largest in Gainesville, Urbana, and Champaign (about 2
  percentage points less mean error), markets where the baseline was weak.
  The text features made errors worse in State College, Savoy, and
  Charlottesville.

3-credit students stop here. 4-credit students continue to Part 3, where we
bring in the list price from the price-history event log and ask how well we
can predict the realized sale price once the list price is known.